# 02 - Library Baseline: better_profanity

Evaluate the `better_profanity` library as a rule-based baseline for Bangla toxic comment detection.

In [ ]:
import time
from better_profanity import profanity
import sys
sys.path.append('..')
from src.dataset import load_dataset
from src.evaluate import get_classification_report
from src.results import build_evaluation, save_evaluation
from src.utils import load_config, ensure_dirs

config = load_config('../configs/config.yaml')
ensure_dirs('../results', '../plots')

## Load Test Data

In [ ]:
DATA_PATH = '../' + config['data']['train_path']

# Same stratified split as the fine-tuning notebooks, so every model sees the same test set
_, texts, _, labels = load_dataset(
    DATA_PATH,
    text_column=config['data']['text_column'],
    label_column=config['data']['label_column'],
    test_size=config['data']['test_size'],
    random_seed=config['data']['random_seed'],
)
print(f'Loaded {len(texts)} test samples')

## Run better_profanity

This library is English-focused, so it serves as a lower-bound baseline showing why Bangla-specific models are needed.

In [ ]:
profanity.load_censor_words()

start_time = time.time()
predictions = [1 if profanity.contains_profanity(text) else 0 for text in texts]
inference_time = time.time() - start_time

print(f'Inference time: {inference_time:.2f}s')
print(f'Samples/second: {len(texts) / inference_time:.0f}')
print(f'Predicted toxic: {sum(predictions)} / {len(predictions)}')

## Evaluation

In [ ]:
evaluation = build_evaluation(labels, predictions, inference_time)

print('=== better_profanity Baseline Results ===')
for k, v in evaluation['metrics'].items():
    print(f'  {k}: {v:.4f}')

print('\n=== Classification Report ===')
print(get_classification_report(labels, predictions))

print('\n=== Confusion Matrix ===')
print(evaluation['confusion_matrix'])

In [ ]:
path = save_evaluation(evaluation, '../results', 'better_profanity')
print(f'Results saved to {path}')